<a href="https://colab.research.google.com/github/asve06/act1_2p_si_eda_2_25_vega/blob/main/code/act2_2p_si_split_2_25_vega.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Women Perfume Recommendation Based on Note Similarities Project**
Ashley Vega   
*Task:* Recommend similar perfumes based on their descriptions.   
*Description:*  The goal of this project is to build a recommendation system that suggests perfumes with similar characteristics or notes by analyzing text data, focusing only on women’s fragrances. This idea explores how machine learning can be used to understand preferences and find connections between products.  
*Task Type:* Classification.  
*Algorithm:* K-Nearest Neighbors (K-NN).

# **Activity 3: Data splitting - final model**

The following section applies K-Fold and Stratified K-Fold to the dataset as required by the assignment. While the core objective of my project is a TF-IDF based recommendation system model which does not rely on training, labels, or data splitting this supervised classification perspective provides an alternative way to evaluate how much information about mainaccord1 can be extracted from the combined note text. These results are included to demonstrate correct use of cross-validation techniques, while the main recommendation system remains the central component of the project.

In [ ]:
import pandas as pd
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

In [ ]:
df_women = pd.read_csv("/content/df_women.csv")

# Create the unified text column combining notes + accords
df_women["features"] = (
    df_women["Top"] + " " +
    df_women["Middle"] + " " +
    df_women["Base"] + " " +
    df_women["mainaccord1"] + " " +
    df_women["mainaccord2"] + " " +
    df_women["mainaccord3"] + " " +
    df_women["mainaccord4"] + " " +
    df_women["mainaccord5"]
)

In [ ]:
# features (X) and target label (y)
X = df_women["features"]
y = df_women["mainaccord1"]

In [ ]:
(df_women["mainaccord1"].value_counts(normalize=True).max())

0.15258112742942573

In [ ]:
def train_test(X_train_fold, X_test_fold, y_train_fold, y_test_fold):
    # Convert text into TF-IDF vectors for train and test
    vectorizer = TfidfVectorizer(stop_words="english")
    X_train_vec = vectorizer.fit_transform(X_train_fold)
    X_test_vec = vectorizer.transform(X_test_fold)

    # KNN model using 5 neighbors and cosine similarity
    knn = KNeighborsClassifier(n_neighbors=5, metric="cosine")
    knn.fit(X_train_vec, y_train_fold)

    # Predict labels for the test fold
    preds = knn.predict(X_test_vec)

    # Storre accuracy for this fold
    return accuracy_score(y_test_fold, preds)

In [ ]:
# K-Fold Cross-Validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)
scores_kf = []

for train_idx, test_idx in kf.split(X):
    score = train_test(
        X.iloc[train_idx], X.iloc[test_idx],
        y.iloc[train_idx], y.iloc[test_idx]
    )
    scores_kf.append(score)

print("K-Fold accuracies:", scores_kf)
print("K-Fold mean:", sum(scores_kf)/len(scores_kf))

K-Fold accuracies: [0.3912087912087912, 0.40369393139841686, 0.3997361477572559, 0.3922603342128408, 0.4050131926121372]
K-Fold mean: 0.3983824794378884


In [ ]:
# Stratified K-Fold Cross-Validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores_skf = []

for train_idx, test_idx in skf.split(X, y):
    score = train_test(
        X.iloc[train_idx], X.iloc[test_idx],
        y.iloc[train_idx], y.iloc[test_idx]
    )
    scores_skf.append(score)

print("Stratified K-Fold accuracies:", scores_skf)
print("Stratified mean:", sum(scores_skf)/len(scores_skf))

/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


Stratified K-Fold accuracies: [0.41186813186813187, 0.3992963940193492, 0.40281442392260336, 0.40325417766051014, 0.3988566402814424]
Stratified mean: 0.4032179535504074


# **Final Model**

This section contains the final model that was obtained at the Activity 2, categorical data encoding and feature scaling techniques, which reflects the actual purpose of this project. The final model is the TF-IDF + KNN recommendation system based on note similarity rather than supervised prediction.

In [ ]:
from sklearn.neighbors import NearestNeighbors

In [ ]:
# TF-IDF Vectorization - features' convertion from text data to a numerical matrix
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(df_women["features"])

# KNN Model using Cosine Distance - measure similarity between text documents
knn = NearestNeighbors(metric='cosine')
knn.fit(tfidf_matrix)

NearestNeighbors(metric='cosine')

In [ ]:
def perfume_recommendation(name, n=5):
    name = name.lower().strip()

    matches = df_women[df_women["Perfume"].str.contains(name, case=False)]
    if matches.empty:
        print(f"No existe el perfume '{name}' en el dataset.")
        return

    # Get the index of the matching perfume.
    idx = matches.index[0]
    base_perfume = df_women.iloc[idx]

    # Find the n+1 nearest neighbors based on the TF-IDF vectors including the base perfume itself
    distances, indices = knn.kneighbors(tfidf_matrix[idx], n_neighbors=n+1)

    print(f"\n🔍 Perfume recommendations for: '{base_perfume['Perfume'].title()}' "
          f"by {base_perfume['Brand'].title()}\n")

    # Iterate through the results, starting from index 1 (to skip the base perfume).
    for i in range(1, n+1):
        sim_idx = indices[0][i]
        perfume = df_women.iloc[sim_idx]

        similarity = (1 - distances[0][i]) * 100

        base_text = df_women.loc[idx, ["Top","Middle","Base"]].str.cat(sep=" ")
        comp_text = df_women.loc[sim_idx, ["Top","Middle","Base"]].str.cat(sep=" ")

        print(f"- {perfume['Perfume'].title()} | Brand: {perfume['Brand'].title()} "
              f"| Similarity: {similarity:.2f}%")

In [ ]:
# Sample of perfume names available in the datasetsample of perfume names available in the dataset
df_women["Perfume"].sample(15).str.title()

,Perfume
8041,Precious-Oud
2466,Flower-By-Kenzo-L-Eau-Originelle
4869,Une-Folie-De-Rose
3772,James-Bond-007-For-Women
7830,Too-Much
2765,Reve-Elixir
6447,Madame
7861,Ultraviolet-Summer-Pop
8516,Cabotine-Sensuelle
9544,Vanille-Pamplemousse


In [ ]:
perfume_recommendation("la-petite-robe-noire")


🔍 Perfume recommendations for: 'La-Petite-Robe-Noire-2' by Guerlain

- Mademoiselle-Guerlain | Brand: Guerlain | Similarity: 63.80%
- Insolence-Limited-Edition | Brand: Guerlain | Similarity: 55.93%
- Insolence | Brand: Guerlain | Similarity: 54.66%
- Married | Brand: Franck-Boclet | Similarity: 49.51%
- Insolence-Shimmering-Edition | Brand: Guerlain | Similarity: 49.33%


In [ ]:
perfume_recommendation("Flowerbomb")


🔍 Perfume recommendations for: 'Flowerbomb-Bomblicious' by Viktor-Rolf

- Cool-Water-Sea-Rose-Exotic-Summer | Brand: Davidoff | Similarity: 73.95%
- La-Mia-Perla-Nera | Brand: La-Perla | Similarity: 69.31%
- Coquette | Brand: Faberlic | Similarity: 64.76%
- Rosa-Damascena | Brand: Granado | Similarity: 64.04%
- Davidoff-Cool-Water-Woman-Sea-Rose-Caribbean-Summer-Edition | Brand: Davidoff | Similarity: 63.76%
